# Exploration de Caltech-101

Objectif : comprendre les catégories, les dimensions et quelques exemples avant de construire le système de recherche.

Dans le terminal, installe les dépendances du notebook si nécessaire : `python -m pip install notebook pillow`. Ouvre ce notebook depuis le dossier du projet et sélectionne le même environnement Python.

**Installation et vérification du noyau**

- `%pip install pillow` installe le paquet **Pillow** dans l’environnement du noyau qui exécute la cellule. Ensuite, Python l’importe sous le nom `PIL`. Le nom du paquet à installer et celui du module à importer sont donc différents.
- `import sys` donne accès aux informations du processus Python. `print(sys.executable)` affiche le chemin exact de l’interpréteur. Si tu as sélectionné le noyau du projet dans VS Code, il doit se terminer par `.venv\Scripts\python.exe`.
- Si le chemin pointe ailleurs, change le noyau du notebook avant de continuer. Activer le venv dans un terminal séparé ne change pas le noyau déjà lancé.

In [ ]:
%pip install pillow
import sys
print(sys.executable)

**Localiser les JPEG**

- `Path("data/101_ObjectCategories")` construit un chemin à partir du répertoire de travail du noyau. `is_dir()` s’assure que la base a été extraite au bon endroit. En cas d’erreur, `resolve()` affiche le chemin absolu où Python a cherché.
- `iterdir()` liste le contenu direct de la base. Le filtre `p.is_dir()` garde les dossiers, dont chacun représente une catégorie dans cette organisation des fichiers.
- Le motif `*/*.jpg` cherche un JPEG à exactement un niveau sous la base. `sorted()` fixe l’ordre des résultats et l’assertion détecte une extraction vide ou mal organisée. Les deux listes servent ensuite à compter les dossiers et à parcourir les fichiers.

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image, ImageOps
from IPython.display import display

base = Path("data/101_ObjectCategories")
if not base.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {base.resolve()}. Ouvre le notebook depuis le dossier du projet.")

categories = sorted(p for p in base.iterdir() if p.is_dir())
images = sorted(base.glob("*/*.jpg"))
assert categories and images, "Aucune catégorie ou image JPEG trouvée."
print(f"Dossiers : {len(categories)}")
print(f"Images JPEG : {len(images)}")

## Nombre d’images par catégorie

Le dossier `BACKGROUND_Google` est conservé dans cette exploration ; son exclusion pour la recherche reste à décider.

**Compter les images par catégorie**

- Pour un chemin comme `airplanes/image_0001.jpg`, `p.parent.name` retourne `airplanes`. Le générateur applique cette opération à chaque JPEG, puis `Counter` ajoute une unité au compteur de sa catégorie.
- `comptes.items()` fournit les couples catégorie/effectif. `sorted()` les ordonne par nom. Les formats `:25` et `:4` alignent les colonnes imprimées.
- `min` et `max` donnent les effectifs extrêmes. Cet écart compte pour la future évaluation, car une moyenne prise sur toutes les requêtes peut donner plus de poids aux grandes catégories. `get("BACKGROUND_Google", 0)` permet aussi de calculer le total hors fond sans modifier les fichiers.

In [ ]:
comptes = Counter(p.parent.name for p in images)
for nom, nombre in sorted(comptes.items()):
    print(f"{nom:25} {nombre:4}")
print(f"\nPlus petite catégorie : {min(comptes.values())} images")
print(f"Plus grande catégorie : {max(comptes.values())} images")
print(f"Sans BACKGROUND_Google : {len(images) - comptes.get('BACKGROUND_Google', 0)} images")

## Dimensions et fichiers illisibles

On ouvre chaque image pour relever ses dimensions et vérifier son décodage. Cette étape peut prendre quelques minutes.

**Vérifier le décodage et mesurer les tailles**

- `Image.open` ouvre chaque fichier dans un bloc `with`, qui libère le fichier dès la fin du bloc. Comme Pillow peut différer la lecture des pixels, `image.load()` force le décodage complet avant de compter le fichier comme lisible.
- `image.size` retourne un couple **(largeur, hauteur)**. Le compteur garde le nombre d’images pour chaque couple et `most_common(10)` affiche les dix tailles les plus fréquentes. Des tailles différentes motivent le redimensionnement de l’étape 02.
- En cas d’erreur de lecture, on conserve le chemin et le message pour retrouver le fichier. L’assertion réconcilie le nombre de succès et d’échecs avec la liste initiale.

In [ ]:
dimensions = Counter()
illisibles = []
for chemin in images:
    try:
        with Image.open(chemin) as image:
            image.load()
            dimensions[image.size] += 1
    except (OSError, ValueError) as erreur:
        illisibles.append((chemin, str(erreur)))

assert sum(dimensions.values()) + len(illisibles) == len(images)
print(f"Images lisibles : {sum(dimensions.values())}")
print(f"Images illisibles : {len(illisibles)}")
print(f"Dimensions distinctes : {len(dimensions)}")
print("Dimensions les plus fréquentes (largeur, hauteur) :")
for taille, nombre in dimensions.most_common(10):
    print(f"  {taille} : {nombre}")
for chemin, erreur in illisibles:
    print(f"Illisible : {chemin} — {erreur}")

## Quelques exemples

Une image de chaque catégorie sélectionnée est affichée avec son nom. Les images originales ne sont pas modifiées.

**Afficher des exemples sans modifier les originaux**

- La boucle choisit cinq dossiers, dont celui de fond. Pour chaque dossier, elle trie les JPEG et prend le premier. Cela donne un aperçu reproductible de la base, avec un seul exemple par dossier.
- `convert("RGB")` met l’image dans un mode d’affichage commun. `ImageOps.contain` réduit ou ajuste l’aperçu pour le faire tenir dans une boîte de 300 × 220 pixels en conservant les proportions. `image.size` donne toujours la taille du fichier ouvert, avant cet ajustement.
- Si un dossier est vide ou que le premier fichier ne s’ouvre pas, la cellule affiche un message et poursuit la boucle.

In [ ]:
for nom in ["airplanes", "Motorbikes", "Faces", "watch", "BACKGROUND_Google"]:
    exemples = sorted((base / nom).glob("*.jpg"))
    if not exemples:
        print(f"Aucune image trouvée pour {nom}")
        continue
    try:
        with Image.open(exemples[0]) as image:
            apercu = ImageOps.contain(image.convert("RGB"), (300, 220))
            print(f"{nom} — {exemples[0].name} — taille originale : {image.size}")
            display(apercu)
    except (OSError, ValueError) as erreur:
        print(f"Impossible d’ouvrir {exemples[0]} : {erreur}")

## À retenir

- Les catégories sont de tailles différentes : il faudra en tenir compte dans l’évaluation.
- Les tailles d’images peuvent varier : le prétraitement devra les harmoniser.
- La catégorie de fond n’est pas une catégorie d’objet ; son traitement sera documenté avant l’évaluation.